In [1]:
import os, sys
import torch
sys.path.append(os.pardir)

from chapter5.gpt_train import optimizer


from chapter5.gpt_download import download_and_load_gpt2
from previous_chapters import GPTModel
from previous_chapters import load_weights_into_gpt

BASE_CONFIG = {
    "vocab_size": 50257,
    "context_length": 1024,
    "drop_rate": 0.0,
    "qkv_bias": True,
}
model_configs = {
    "gpt2-medium (355M)": {"emb_dim": 1024, "n_layers": 24, "n_heads": 16}
}
CHOOSE_MODEL = "gpt2-medium (355M)"
BASE_CONFIG.update(model_configs[CHOOSE_MODEL])
model_size = CHOOSE_MODEL.split(" ")[-1].lstrip("(").rstrip(")")
settings, params = download_and_load_gpt2(
    model_size=model_size,
    models_dir="gpt2"
)
model = GPTModel(BASE_CONFIG)
load_weights_into_gpt(model, params)
model.eval()

ModuleNotFoundError: No module named 'chapter5.gpt_train'

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if torch.backends.mps.is_available():
    device = torch.device("mps")

In [ ]:
from previous_chapters import (
    calc_loss_loader,
    train_model_simple
)
import torch
from data_loader import train_loader, val_loader, test_loader

In [ ]:
model.to("cpu")
with torch.no_grad():
    train_loss = calc_loss_loader(train_loader, model, device=device, num_batches=5)
    val_loss = calc_loss_loader(val_loader, model, device=device, num_batches=5)

In [ ]:
print("train loss: ", train_loss)
print("val loss: ", val_loss)

In [7]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")

In [8]:
from data_loader import val_data
from prompt import format_input

In [ ]:
import time

start_time = time.time()
torch.manual_seed(123)
optimizer = torch.optim.AdamW(
    model.parameters(), lr=0.00005, weight_decay=0.1
)
num_epochs = 2

train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device=device, num_epochs=num_epochs, eval_freq=5, eval_iter=5, start_context=format_input(val_data[0]), tokenizer=tokenizer
)

Ep 1 (Step 000000): Train loss 2.776, Val loss 2.799
Ep 1 (Step 000005): Train loss 1.207, Val loss 1.106
Ep 1 (Step 000010): Train loss 0.872, Val loss 0.926
Ep 1 (Step 000015): Train loss 0.856, Val loss 0.858
Ep 1 (Step 000020): Train loss 0.788, Val loss 0.839
Ep 1 (Step 000025): Train loss 0.775, Val loss 0.805
Ep 1 (Step 000030): Train loss 0.801, Val loss 0.785
Ep 1 (Step 000035): Train loss 0.716, Val loss 0.763
Ep 1 (Step 000040): Train loss 0.669, Val loss 0.760
Ep 1 (Step 000045): Train loss 0.634, Val loss 0.759
Ep 1 (Step 000050): Train loss 0.663, Val loss 0.747
Ep 1 (Step 000055): Train loss 0.764, Val loss 0.725
Ep 1 (Step 000060): Train loss 0.720, Val loss 0.705
Ep 1 (Step 000065): Train loss 0.651, Val loss 0.706
Ep 1 (Step 000070): Train loss 0.531, Val loss 0.705
Ep 1 (Step 000075): Train loss 0.566, Val loss 0.704
Ep 1 (Step 000080): Train loss 0.604, Val loss 0.706
Ep 1 (Step 000085): Train loss 0.511, Val loss 0.689
Ep 1 (Step 000090): Train loss 0.565, Val loss

In [ ]:
end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Execution time: {execution_time_minutes:.2f} minutes")

In [ ]:
torch.save(model.state_dict(), "gpt2_instruction_finetunned.pth")